# Brainana — NHP Anatomical Volume Processing

Full anatomical pipeline for non-human primate T1w MRI. Outputs are BIDS derivatives.

1. Conform to template space (antsAI global rotation search + ANTs rigid)
2. Segmentation + skull stripping (`fastsurfer_nn` on conformed image → ARM2 + brain mask)
3. N4 bias field correction
4. Registration to template (FireANTs SyN on GPU, antspyx SyN on CPU)
5. Backprojection of all atlases to T1w and scanner space

Surface reconstruction not included (requires FreeSurfer license).

---
**Local:** `RUN_LOCAL = True`. Place notebook, `brainana_anat.zip`, and T1w in same folder.

**Colab:** `RUN_LOCAL = False`. Place notebook, `brainana_anat.zip`, and T1w in same Google Drive folder. Set runtime to T4 GPU.

## Cell 1 — Setup & User Settings

In [1]:
import shutil
import os
import sys
import zipfile
import subprocess
import re
from pathlib import Path

# ════════════════════════════════════════════════════════════════
# USER SETTINGS — edit these before running
# ════════════════════════════════════════════════════════════════

RUN_LOCAL  = True           # True = local Jupyter, False = Google Colab

DRIVE_DIR  = '/content/drive/MyDrive/Colab Notebooks/BrainanaAnat'
                            # Colab only: Google Drive folder

TEMPLATE   = 'NMT2Sym'     # options: 'NMT2Sym', 'NMT2Asym', 'MEBRAINS', 'Yerkes19', 'D99'

USE_GPU    = True           # set to False if no GPU available

SUBJECT_ID = 'baby4'           # e.g. 'baby4' — if None, auto-detected from T1w filename
SESSION_ID = None           # e.g. '01'    — if None, no session level in output

# ════════════════════════════════════════════════════════════════

if RUN_LOCAL:
    WORK_DIR     = os.getcwd()
    PACKAGE_DIR  = os.path.join(WORK_DIR, 'brainana_anat')
    OUTPUT_DIR   = os.path.join(WORK_DIR, 'brainana_anat_output')
    EXTRACT_ROOT = WORK_DIR
    print(f'Running locally from: {WORK_DIR}')
else:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK_DIR     = DRIVE_DIR
    PACKAGE_DIR  = '/content/brainana_anat'
    OUTPUT_DIR   = '/content/brainana_anat_output'
    EXTRACT_ROOT = '/content'
    print(f'Running on Colab from: {WORK_DIR}')

# Install dependencies
subprocess.run(['pip', 'install', '-q',
                'antspyx', 'nibabel', 'nilearn', 'matplotlib',
                'torch', 'torchvision', 'pyyaml', 'yacs',
                'h5py', 'pandas', 'scipy', 'seaborn',
                'scikit-image', 'scikit-learn', 'Pillow',
                'pybids', 'packaging', 'psutil', 'requests',
                'torchio', 'tqdm', 'tensorboard'])

# FireANTs
import torch
FIREANTS_DIR = os.path.join(os.path.expanduser('~'), 'fireants') if RUN_LOCAL else '/content/fireants'
if not os.path.exists(FIREANTS_DIR):
    subprocess.run(['git', 'clone', '--quiet', 'https://github.com/rohitrango/fireants', FIREANTS_DIR])
    setup_cfg = os.path.join(FIREANTS_DIR, 'setup.py')
    with open(setup_cfg, 'r') as f:
        c = f.read()
    with open(setup_cfg, 'w') as f:
        f.write(c.replace('simpleitk==2.2.1', 'simpleitk'))
    subprocess.run(['pip', 'install', '-q', '.'], cwd=FIREANTS_DIR)
    if torch.cuda.is_available():
        subprocess.run(['python', 'setup.py', 'build_ext'], cwd=os.path.join(FIREANTS_DIR, 'fused_ops'))
        subprocess.run(['python', 'setup.py', 'install'],   cwd=os.path.join(FIREANTS_DIR, 'fused_ops'))
        print('FireANTs installed with fused CUDA ops.')
    else:
        print('FireANTs installed (no CUDA — fused ops skipped).')
else:
    print('FireANTs already installed.')

# Extract package
if os.path.exists(PACKAGE_DIR):
    shutil.rmtree(PACKAGE_DIR)
zip_path = os.path.join(WORK_DIR, 'brainana_anat.zip')
if not os.path.exists(zip_path):
    raise FileNotFoundError(f'brainana_anat.zip not found in {WORK_DIR}')
print(f'Extracting from: {zip_path}')
with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall(EXTRACT_ROOT)
print('Done.')

# Patch snapshots.py
snapshots_py = os.path.join(PACKAGE_DIR, 'src/nhp_mri_prep/quality_control/snapshots.py')
with open(snapshots_py, 'r') as f:
    c = f.read()
c = c.replace(
    'from fastsurfer_surfrecon.io.surface import convert_fs_surface_to_gifti',
    'try:\n    from fastsurfer_surfrecon.io.surface import convert_fs_surface_to_gifti\nexcept Exception:\n    convert_fs_surface_to_gifti = None'
)
with open(snapshots_py, 'w') as f:
    f.write(c)
print('Patches applied.')

sys.path.insert(0, os.path.join(PACKAGE_DIR, 'src'))

# antsAI — ships with ANTs/antspyx, handles large rotations via global search
import shutil as _shutil
ANTS_AI = _shutil.which('antsAI')
print(f'antsAI: {ANTS_AI or "not found — will use convergencePoint init"}')

# Template lookup
TEMPLATE_LOOKUP = {
    'NMT2Sym':  {'head': 'tpl-NMT2Sym_res-05_T1w.nii.gz',  'brain': 'tpl-NMT2Sym_res-025_T1w_brain.nii.gz', 'dir': ''},
    'NMT2Asym': {'head': 'tpl-NMT2Asym_res-05_T1w.nii.gz', 'brain': 'tpl-NMT2Asym_res-05_T1w_brain.nii.gz', 'dir': 'NMT2Asym'},
    'MEBRAINS': {'head': 'tpl-MEBRAINS_res-04_T1w.nii.gz',  'brain': 'tpl-MEBRAINS_res-04_T1w_brain.nii.gz', 'dir': 'MEBRAINS'},
    'Yerkes19': {'head': 'tpl-Yerkes19_res-05_T1w.nii.gz',  'brain': 'tpl-Yerkes19_res-05_T1w_brain.nii.gz', 'dir': 'Yerkes19'},
    'D99':      {'head': 'tpl-D99_res-05_T1w_brain.nii.gz', 'brain': 'tpl-D99_res-05_T1w_brain.nii.gz',      'dir': 'D99'},
}
if TEMPLATE not in TEMPLATE_LOOKUP:
    raise ValueError(f'Unknown template: {TEMPLATE}. Choose from {list(TEMPLATE_LOOKUP.keys())}')
tmpl_cfg       = TEMPLATE_LOOKUP[TEMPLATE]
TMPL_DIR       = os.path.join(PACKAGE_DIR, 'templates', tmpl_cfg['dir']) if tmpl_cfg['dir'] else os.path.join(PACKAGE_DIR, 'templates')
TEMPLATE_HEAD  = os.path.join(TMPL_DIR, tmpl_cfg['head'])
TEMPLATE_BRAIN = os.path.join(TMPL_DIR, tmpl_cfg['brain'])

ATLAS_FILES = sorted([
    p for p in Path(PACKAGE_DIR).rglob('*.nii.gz')
    if f'space-{TEMPLATE}' in p.name and 'atlas' in p.name
])

DEVICE       = 'cuda' if (USE_GPU and torch.cuda.is_available()) else 'cpu'
USE_FIREANTS = (DEVICE == 'cuda')

print(f'\nMode:      {"Local" if RUN_LOCAL else "Colab"}')
print(f'TEMPLATE:  {TEMPLATE}')
print(f'Device:    {DEVICE}')
print(f'FireANTs:  {USE_FIREANTS}')
print(f'Atlases:   {len(ATLAS_FILES)}')
print('\nSetup complete.')

Running locally from: /Users/marcaro/Library/CloudStorage/GoogleDrive-michael.j.arcaro@gmail.com/My Drive/Colab Notebooks/BrainanaLite


FireANTs already installed.
Extracting from: /Users/marcaro/Library/CloudStorage/GoogleDrive-michael.j.arcaro@gmail.com/My Drive/Colab Notebooks/BrainanaLite/brainana_anat.zip
Done.
Patches applied.
antsAI: /Users/marcaro/ANTs/bin/antsAI

Mode:      Local
TEMPLATE:  NMT2Sym
Device:    cpu
FireANTs:  False
Atlases:   14

Setup complete.


## Cell 2 — Load T1w image & set BIDS identifiers

In [3]:
import nibabel as nib
import numpy as np

t1_candidates = [p for p in Path(WORK_DIR).glob('*.nii*')
                 if 'T1' in p.name or 't1' in p.name]
if not t1_candidates:
    t1_candidates = list(Path(WORK_DIR).glob('*.nii*'))
if not t1_candidates:
    raise FileNotFoundError(f'No NIfTI files found in {WORK_DIR}')

T1_PATH = str(t1_candidates[0])
t1_img  = nib.load(T1_PATH)
t1_name = Path(T1_PATH).name

if SUBJECT_ID is None:
    m = re.search(r'sub-([^_]+)', t1_name)
    SUBJECT_ID = m.group(1) if m else 'unknown'
if SESSION_ID is None:
    m = re.search(r'ses-([^_]+)', t1_name)
    SESSION_ID = m.group(1) if m else None

BIDS_PREFIX = f'sub-{SUBJECT_ID}'
if SESSION_ID:
    BIDS_PREFIX += f'_ses-{SESSION_ID}'
    ANAT_DIR = os.path.join(OUTPUT_DIR, f'sub-{SUBJECT_ID}', f'ses-{SESSION_ID}', 'anat')
    FIG_DIR  = os.path.join(OUTPUT_DIR, f'sub-{SUBJECT_ID}', f'ses-{SESSION_ID}', 'figures')
else:
    ANAT_DIR = os.path.join(OUTPUT_DIR, f'sub-{SUBJECT_ID}', 'anat')
    FIG_DIR  = os.path.join(OUTPUT_DIR, f'sub-{SUBJECT_ID}', 'figures')

os.makedirs(ANAT_DIR, exist_ok=True)
os.makedirs(FIG_DIR,  exist_ok=True)
os.makedirs(os.path.join(ANAT_DIR, 'atlas_space-T1w'),    exist_ok=True)
os.makedirs(os.path.join(ANAT_DIR, 'atlas_space-scanner'), exist_ok=True)

print(f'T1w:         {T1_PATH}')
print(f'Shape:       {t1_img.shape}')
print(f'Voxel size:  {np.round(t1_img.header.get_zooms(), 3)} mm')
print(f'Subject:     {SUBJECT_ID}')
print(f'Session:     {SESSION_ID or "none"}')
print(f'BIDS prefix: {BIDS_PREFIX}')

if len(t1_candidates) > 1:
    print(f'\nMultiple NIfTI files found — using: {T1_PATH}')
    print('Others:', [str(p) for p in t1_candidates[1:]])

T1w:         /Users/marcaro/Library/CloudStorage/GoogleDrive-michael.j.arcaro@gmail.com/My Drive/Colab Notebooks/BrainanaLite/baby4_mean_anat.nii
Shape:       (256, 256, 208)
Voxel size:  [0.5 0.5 0.5] mm
Subject:     baby4
Session:     none
BIDS prefix: sub-baby4


## Cell 3 — Load models

In [5]:
import yaml
import ants

SRC_DIR         = os.path.join(PACKAGE_DIR, 'src')
SKULLSTRIP_SRC  = os.path.join(SRC_DIR, 'nhp_skullstrip_nn')
FASTSURFER_SRC  = os.path.join(SRC_DIR, 'fastsurfer_nn')
SKULLSTRIP_CKPT = os.path.join(SKULLSTRIP_SRC, 'pretrained_model', 'T1w_brainmask.pth')
FASTSURFER_CKPT = os.path.join(FASTSURFER_SRC, 'pretrained_model')

ckpt_yaml = os.path.join(FASTSURFER_SRC, 'config', 'checkpoint_paths.yaml')
with open(ckpt_yaml, 'r') as f:
    ckpt_cfg = yaml.safe_load(f)
for key in list(ckpt_cfg.keys()):
    fname = os.path.basename(str(ckpt_cfg[key]))
    local = os.path.join(FASTSURFER_CKPT, fname)
    if os.path.exists(local):
        ckpt_cfg[key] = local
with open(ckpt_yaml, 'w') as f:
    yaml.dump(ckpt_cfg, f)
print('Checkpoint paths patched.')

from nhp_skullstrip_nn.model.model_loader import ModelLoader
skullstrip_model = ModelLoader.load_model_from_file(SKULLSTRIP_CKPT, device_id=DEVICE)
skullstrip_model.eval()
print('nhp_skullstrip_nn loaded.')

from fastsurfer_nn.inference.api import segmentation as fastsurfer_segment
print('fastsurfer_nn ready.')

Checkpoint paths patched.
nhp_skullstrip_nn loaded.
fastsurfer_nn ready.


## Cell 4 — Conform to template space
_antsAI performs a global rotation search to find the best initial alignment,_
_then ANTs rigid refines it. Handles large rotations robustly without FLIRT._

In [7]:
print('Conforming T1w to template grid...')

from nhp_skullstrip_nn.inference.prediction import skullstripping

t1_ants    = ants.image_read(T1_PATH)
tmpl_brain = ants.image_read(TEMPLATE_BRAIN)
tmpl_head  = ants.image_read(TEMPLATE_HEAD)

# Step 1: skull strip T1w to improve registration quality
_tmp_dir   = os.path.join(OUTPUT_DIR, '_tmp_conform')
os.makedirs(_tmp_dir, exist_ok=True)
_tmp_mask  = os.path.join(_tmp_dir, 'conform_mask.nii.gz')
_tmp_brain = os.path.join(_tmp_dir, 'conform_brain.nii.gz')

skullstripping(input_image=T1_PATH, modal='anat',
               output_path=_tmp_mask, device_id=DEVICE)
_mask_for_conform  = ants.image_read(_tmp_mask)
_brain_for_conform = t1_ants * _mask_for_conform
ants.image_write(_brain_for_conform, _tmp_brain)

# Step 2: antsAI global rotation search → best initial transform
_init_xfm = os.path.join(_tmp_dir, 'init_transform.mat')
if ANTS_AI:
    print('Running antsAI global rotation search...')
    subprocess.run([
        ANTS_AI, '-d', '3',
        '-m', f'Mattes[{TEMPLATE_BRAIN},{_tmp_brain},32,Regular,0.2]',
        '-t', 'Rigid[0.1]',
        '-s', '20[0,1]',
        '-p', '0',
        '-o', _init_xfm,
    ], check=True)
    init_tx = _init_xfm
    print('antsAI complete.')
else:
    print('WARNING: antsAI not found — using convergencePoint init (may fail for large rotations)')
    init_tx = 'convergencePoint'

# Step 3: ANTs rigid refinement from global optimum
conform_reg = ants.registration(
    fixed             = tmpl_brain,
    moving            = ants.image_read(_tmp_brain),
    type_of_transform = 'Rigid',
    initial_transform = init_tx,
    verbose           = False
)

# Step 4: apply transform to FULL HEAD T1w
t1_conformed = ants.apply_transforms(
    fixed         = tmpl_brain,
    moving        = t1_ants,
    transformlist = conform_reg['fwdtransforms'],
    interpolator  = 'linear'
)

CONFORMED_PATH = os.path.join(ANAT_DIR, f'{BIDS_PREFIX}_space-T1w_desc-conformed_T1w.nii.gz')
ants.image_write(t1_conformed, CONFORMED_PATH)

import shutil as _shutil
_shutil.rmtree(_tmp_dir)
print(f'Saved: {CONFORMED_PATH}')

2026-05-22 13:35:29 | INFO     | Starting skullstripping for anat modality using nhp_skullstrip_nn
2026-05-22 13:35:29 | INFO     | Using model: /Users/marcaro/Library/CloudStorage/GoogleDrive-michael.j.arcaro@gmail.com/My Drive/Colab Notebooks/BrainanaLite/brainana_anat/src/nhp_skullstrip_nn/pretrained_model/T1w_brainmask.pth
2026-05-22 13:35:29 | INFO     | Loading model from: /Users/marcaro/Library/CloudStorage/GoogleDrive-michael.j.arcaro@gmail.com/My Drive/Colab Notebooks/BrainanaLite/brainana_anat/src/nhp_skullstrip_nn/pretrained_model/T1w_brainmask.pth on device: cpu
2026-05-22 13:35:29 | INFO     | Loading checkpoint...
2026-05-22 13:35:29 | INFO     | Extracting config from checkpoint
2026-05-22 13:35:29 | INFO     | Using 'model_state_dict' from checkpoint
2026-05-22 13:35:29 | INFO     | Model loaded


Conforming T1w to template grid...


2026-05-22 13:35:55 | INFO     | Skullstripping completed successfully


Running antsAI global rotation search...
antsAI complete.
Saved: /Users/marcaro/Library/CloudStorage/GoogleDrive-michael.j.arcaro@gmail.com/My Drive/Colab Notebooks/BrainanaLite/brainana_anat_output/sub-baby4/anat/sub-baby4_space-T1w_desc-conformed_T1w.nii.gz


## Cell 5 — Segmentation + skull stripping
_fastsurfer_nn on conformed full-head image. Brain mask from segmentation._

In [15]:
print('Running segmentation (fastsurfer_nn)...', flush=True)

import os
import gc
import shutil as _shutil
from pathlib import Path
import numpy as np
import ants

SEG_OUTPUT_DIR = os.path.join(OUTPUT_DIR, '_fastsurfer_seg')
os.makedirs(SEG_OUTPUT_DIR, exist_ok=True)

ckpt_ax  = Path(os.path.join(FASTSURFER_CKPT, 'T1w_seg-ARM2_axial.pkl'))
ckpt_cor = Path(os.path.join(FASTSURFER_CKPT, 'T1w_seg-ARM2_coronal.pkl'))
ckpt_sag = Path(os.path.join(FASTSURFER_CKPT, 'T1w_seg-ARM2_sagittal.pkl'))

print('Checking FastSurfer checkpoint files...', flush=True)
for ckpt in [ckpt_ax, ckpt_cor, ckpt_sag]:
    if not ckpt.exists():
        raise FileNotFoundError(f'Missing checkpoint: {ckpt}')
    print(f'  found: {ckpt.name}', flush=True)

if not os.path.exists(CONFORMED_PATH):
    raise FileNotFoundError(f'Conformed image not found: {CONFORMED_PATH}')

print(f'Input image: {CONFORMED_PATH}', flush=True)
print(f'Output dir:  {SEG_OUTPUT_DIR}', flush=True)
print(f'Device:      {DEVICE}', flush=True)
print('Starting fastsurfer_segment. This may take a while on CPU...', flush=True)

result = fastsurfer_segment(
    input_image           = CONFORMED_PATH,
    output_dir            = SEG_OUTPUT_DIR,
    ckpt_ax               = ckpt_ax,
    ckpt_cor              = ckpt_cor,
    ckpt_sag              = ckpt_sag,
    device                = DEVICE,
    output_data_format    = 'nifti',
    plane_weight_coronal  = 0.4,
    plane_weight_axial    = 0.4,
    plane_weight_sagittal = 0.2,
    batch_size            = 1 if DEVICE == 'cpu' else 8
)

print('fastsurfer_segment finished.', flush=True)
print(f'Result type: {type(result)}', flush=True)

if isinstance(result, dict):
    print(f'Result keys: {list(result.keys())}', flush=True)
else:
    raise TypeError(f'Expected fastsurfer_segment to return dict, got {type(result)}')

SEG_PATH  = os.path.join(ANAT_DIR, f'{BIDS_PREFIX}_space-T1w_atlas-ARM2_dseg.nii.gz')
MASK_PATH = os.path.join(ANAT_DIR, f'{BIDS_PREFIX}_space-T1w_desc-brain_mask.nii.gz')
BRAIN_PATH = os.path.join(ANAT_DIR, f'{BIDS_PREFIX}_space-T1w_desc-preproc_brain.nii.gz')

if 'segmentation' not in result:
    raise KeyError(f"fastsurfer_segment result missing 'segmentation'. Keys: {list(result.keys())}")

seg_src = str(result['segmentation'])
if not os.path.exists(seg_src):
    raise FileNotFoundError(f'Segmentation output reported but not found: {seg_src}')

_shutil.copy(seg_src, SEG_PATH)
print(f'Saved segmentation: {SEG_PATH}', flush=True)

if 'mask' in result and result['mask'] is not None and os.path.exists(str(result['mask'])):
    mask_src = str(result['mask'])
    _shutil.copy(mask_src, MASK_PATH)
    print(f'Saved brain mask from fastsurfer output: {MASK_PATH}', flush=True)
else:
    print("No usable result['mask'] found. Creating mask from segmentation labels > 0...", flush=True)

    seg_ants = ants.image_read(SEG_PATH)
    seg_arr = seg_ants.numpy()

    mask_arr = (seg_arr > 0).astype(np.float32)
    mask_ants = ants.from_numpy(
        mask_arr,
        origin=seg_ants.origin,
        spacing=seg_ants.spacing,
        direction=seg_ants.direction
    )
    ants.image_write(mask_ants, MASK_PATH)
    print(f'Saved derived brain mask: {MASK_PATH}', flush=True)

mask_ants = ants.image_read(MASK_PATH)
t1_conformed_for_brain = ants.image_read(CONFORMED_PATH)

if mask_ants.shape != t1_conformed_for_brain.shape:
    raise ValueError(
        f'Mask and conformed image shape mismatch: '
        f'mask={mask_ants.shape}, image={t1_conformed_for_brain.shape}'
    )

t1_brain = t1_conformed_for_brain * mask_ants
ants.image_write(t1_brain, BRAIN_PATH)

print(f'Saved brain:        {BRAIN_PATH}', flush=True)

for path_name, path in [
    ('SEG_PATH', SEG_PATH),
    ('MASK_PATH', MASK_PATH),
    ('BRAIN_PATH', BRAIN_PATH),
]:
    if not os.path.exists(path):
        raise FileNotFoundError(f'{path_name} was not created: {path}')
    print(f'Verified {path_name}: {path}', flush=True)

gc.collect()
print('Cell 5 complete. Memory freed.', flush=True)

Running segmentation (fastsurfer_nn)...
Checking FastSurfer checkpoint files...
  found: T1w_seg-ARM2_axial.pkl
  found: T1w_seg-ARM2_coronal.pkl
  found: T1w_seg-ARM2_sagittal.pkl
Input image: /Users/marcaro/Library/CloudStorage/GoogleDrive-michael.j.arcaro@gmail.com/My Drive/Colab Notebooks/BrainanaLite/brainana_anat_output/sub-baby4/anat/sub-baby4_space-T1w_desc-conformed_T1w.nii.gz
Output dir:  /Users/marcaro/Library/CloudStorage/GoogleDrive-michael.j.arcaro@gmail.com/My Drive/Colab Notebooks/BrainanaLite/brainana_anat_output/_fastsurfer_seg
Device:      cpu
Starting fastsurfer_segment. This may take a while on CPU...


100%|██████████████████████████████████████| 200/200 [00:49<00:00,  4.01batch/s]


✓ Loaded WM labels from extended ColorLUT
Checking for disconnected WM islands...
  LH WM (label=-1001) has 45 components
    2 / 45 island (size=183) is in the correct hemisphere
    3 / 45 island (size=53) is in the correct hemisphere
    4 / 45 island (size=77) is in the correct hemisphere
    5 / 45 island (size=5) is in the correct hemisphere
    6 / 45 island (size=4) is in the correct hemisphere
    7 / 45 island (size=10) is in the correct hemisphere
    8 / 45 island (size=1) is in the correct hemisphere
    9 / 45 island (size=5) is in the correct hemisphere
    10 / 45 island (size=1) is in the correct hemisphere
    11 / 45 island (size=1) is in the correct hemisphere
    12 / 45 island (size=1) is in the correct hemisphere
    13 / 45 island (size=6) is in the correct hemisphere
    14 / 45 island (size=2) is in the correct hemisphere
    15 / 45 island (size=1) is in the correct hemisphere
    16 / 45 island (size=19) is in the correct hemisphere
    17 / 45 island (size=

## Cell 6 — N4 bias field correction
_fastsurfer_nn mask, bspline [ 150 ], rescale mean to 100._

In [17]:
print('Running N4 bias field correction...')

t1_n4 = ants.n4_bias_field_correction(
    t1_brain,
    mask          = mask_ants,
    shrink_factor = 2,
    convergence   = {'iters': [50, 50, 30, 20], 'tol': 1e-7},
    spline_param  = 150,
    verbose       = False
)

n4_arr   = t1_n4.numpy()
mask_arr = mask_ants.numpy() > 0
mean_int = n4_arr[mask_arr].mean()
t1_n4    = t1_n4 * (100.0 / mean_int)

N4_PATH = os.path.join(ANAT_DIR, f'{BIDS_PREFIX}_space-T1w_desc-preproc_T1w.nii.gz')
ants.image_write(t1_n4, N4_PATH)
print(f'Saved: {N4_PATH}')

Running N4 bias field correction...
Saved: /Users/marcaro/Library/CloudStorage/GoogleDrive-michael.j.arcaro@gmail.com/My Drive/Colab Notebooks/BrainanaLite/brainana_anat_output/sub-baby4/anat/sub-baby4_space-T1w_desc-preproc_T1w.nii.gz


## Cell 7 — Registration to template
_FireANTs SyN on GPU (~5 min), antspyx SyN on CPU (~30-60 min)._

In [ ]:
print(f'Registration backend: {"FireANTs" if USE_FIREANTS else "antspyx"}')
print(f'Registering to {TEMPLATE}...')

T1_TMPL_PATH   = os.path.join(ANAT_DIR, f'{BIDS_PREFIX}_space-{TEMPLATE}_desc-preproc_T1w.nii.gz')
MASK_TMPL_PATH = os.path.join(ANAT_DIR, f'{BIDS_PREFIX}_space-{TEMPLATE}_desc-brain_mask.nii.gz')

if USE_FIREANTS:
    from fireants.io.image import BatchedImages
    from fireants.registration import SyN
    fixed  = BatchedImages.from_file([TEMPLATE_BRAIN])
    moving = BatchedImages.from_file([N4_PATH])
    reg = SyN(scales=[4,2,1], iterations=[500,300,100],
              fixed_images=fixed, moving_images=moving)
    reg.optimize(save_transformed=True)
    reg.save_transformed(T1_TMPL_PATH)
    
    # Use reg.transform for mask in FireANTs branch
    mask_warped = reg.transform(
        BatchedImages.from_file([MASK_PATH]),
        fixed_images=BatchedImages.from_file([TEMPLATE_BRAIN]),
        inverse=False, interpolation='nearest'
    )
    mask_warped.save(MASK_TMPL_PATH)

else:
    reg = ants.registration(
        fixed                   = tmpl_brain,
        moving                  = t1_n4,
        type_of_transform       = 'SyN',
        initial_transform       = 'identity',
        syn_metric              = 'CC',
        syn_sampling            = 4,
        reg_iterations          = (500, 300, 100),
        aff_metric              = 'mattes',
        aff_sampling            = 32,
        grad_step               = 0.1,
        flow_sigma              = 3,
        total_sigma             = 0,
        winsorize_low_quantile  = 0.005,
        winsorize_high_quantile = 0.995,
        verbose                 = False
    )
    print('Writing registered T1w and mask...')
    
    FWD_TRANSFORMS = reg['fwdtransforms']
    INV_TRANSFORMS = reg['invtransforms']
    
    ants.image_write(
        ants.apply_transforms(
            fixed=tmpl_brain,
            moving=t1_n4,
            transformlist=FWD_TRANSFORMS,
            interpolator='bSpline'
        ),
        T1_TMPL_PATH
    )
    
    ants.image_write(
        ants.apply_transforms(
            fixed=tmpl_brain,
            moving=mask_ants,
            transformlist=FWD_TRANSFORMS,
            interpolator='nearestNeighbor'
        ),
        MASK_TMPL_PATH
    )
    
    print(f'Saved: {T1_TMPL_PATH}')
    print(f'Saved: {MASK_TMPL_PATH}')

Registration backend: antspyx
Registering to NMT2Sym...
Saved: /Users/marcaro/Library/CloudStorage/GoogleDrive-michael.j.arcaro@gmail.com/My Drive/Colab Notebooks/BrainanaLite/brainana_anat_output/sub-baby4/anat/sub-baby4_space-NMT2Sym_desc-preproc_T1w.nii.gz


## Cell 8 — Atlas backprojection
_All atlases use NearestNeighbor — matching full Brainana pipeline._

In [ ]:
print(f'Backprojecting {len(ATLAS_FILES)} atlases...')

for atlas_path in ATLAS_FILES:
    atlas_ants = ants.image_read(str(atlas_path))
    imagetype  = 3 if atlas_ants.dimension == 4 else 0
    m = re.search(r'atlas-([^_]+)', atlas_path.name)
    atlas_name = m.group(1) if m else atlas_path.stem
    out_t1w = os.path.join(ANAT_DIR, 'atlas_space-T1w',
                           f'{BIDS_PREFIX}_space-T1w_atlas-{atlas_name}_dseg.nii.gz')
    out_scanner = os.path.join(ANAT_DIR, 'atlas_space-scanner',
                               f'{BIDS_PREFIX}_space-scanner_atlas-{atlas_name}_dseg.nii.gz')

    if USE_FIREANTS:
        from fireants.io.image import BatchedImages
        warped = reg.transform(
            BatchedImages.from_file([str(atlas_path)]),
            fixed_images=BatchedImages.from_file([N4_PATH]),
            inverse=True, interpolation='nearest'
        )
        warped.save(out_t1w)
    else:
        ants.image_write(
            ants.apply_transforms(t1_n4, atlas_ants, INV_TRANSFORMS,
                                  interpolator='nearestNeighbor',
                                  imagetype=imagetype),
            out_t1w
        )

    warped_t1w_ants = ants.image_read(out_t1w)
    ants.image_write(
        ants.apply_transforms(
            fixed=t1_ants, moving=warped_t1w_ants,
            transformlist=[conform_reg['fwdtransforms'][0]],
            whichtoinvert=[True],
            interpolator='nearestNeighbor',
            imagetype=3 if warped_t1w_ants.dimension == 4 else 0
        ),
        out_scanner
    )
    print(f'  {atlas_name}')

print('\nAtlas backprojection complete.')

## Cell 9 — QC figures

In [ ]:
import logging
from nhp_mri_prep.quality_control import (
    create_conform_qc, create_skullstripping_qc,
    create_atlas_segmentation_qc, create_bias_correction_qc,
    create_registration_qc,
)

qc_logger = logging.getLogger('brainana_anat_qc')

create_conform_qc(
    conformed_file=CONFORMED_PATH, template_file=TEMPLATE_BRAIN,
    save_f=os.path.join(FIG_DIR, f'{BIDS_PREFIX}_desc-conform_T1w.png'),
    modality='anat', logger=qc_logger)
print('Conform QC saved.')

create_skullstripping_qc(
    underlay_file=CONFORMED_PATH, mask_file=MASK_PATH,
    save_f=os.path.join(FIG_DIR, f'{BIDS_PREFIX}_desc-skullstrip_T1w.png'),
    modality='anat', logger=qc_logger)
print('Skull strip QC saved.')

create_atlas_segmentation_qc(
    underlay_file=BRAIN_PATH, seg_file=SEG_PATH, atlas_file=SEG_PATH,
    save_f=os.path.join(FIG_DIR, f'{BIDS_PREFIX}_desc-atlasSegmentation_T1w.png'),
    modality='anat', logger=qc_logger)
print('Segmentation QC saved.')

create_bias_correction_qc(
    image_original=BRAIN_PATH, image_corrected=N4_PATH,
    save_f=os.path.join(FIG_DIR, f'{BIDS_PREFIX}_desc-biascorrect_T1w.png'),
    modality='anat', logger=qc_logger)
print('Bias correction QC saved.')

create_registration_qc(
    image_file=T1_TMPL_PATH, template_file=TEMPLATE_BRAIN,
    save_f=os.path.join(FIG_DIR, f'{BIDS_PREFIX}_desc-anat2template_T1w.png'),
    modality='anat2template', logger=qc_logger)
print('Registration QC saved.')
print(f'\nAll QC figures saved to: {FIG_DIR}')

## Cell 10 — Save outputs

In [ ]:
import shutil

FINAL_OUTPUT = os.path.join(WORK_DIR, 'brainana_anat_output')

if os.path.realpath(FINAL_OUTPUT) != os.path.realpath(OUTPUT_DIR):
    if os.path.exists(FINAL_OUTPUT):
        shutil.rmtree(FINAL_OUTPUT)
    shutil.copytree(OUTPUT_DIR, FINAL_OUTPUT)
    print(f'Outputs copied to: {FINAL_OUTPUT}')
else:
    print(f'Outputs already in place: {OUTPUT_DIR}')

print('\nFiles:')
base = FINAL_OUTPUT if os.path.exists(FINAL_OUTPUT) else OUTPUT_DIR
for root, dirs, fnames in os.walk(base):
    dirs[:] = [d for d in dirs if not d.startswith('_')]
    for fname in fnames:
        print(' ', os.path.relpath(os.path.join(root, fname), base))